# Stress Field Evolution Video
Genera GIFs/MP4s que muestran la evolución de los campos de tensión para N RVEs del master file `rve_run2.h5`.

**Layout por frame:**
```
[ Material Phase  |  Strain Evolution + dot ]   ← fila superior
[   σxx   |   σyy   |   σxy                 ]   ← fila inferior
```

In [1]:
import h5py
import numpy as np
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from pathlib import Path

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────
MASTER_FILE   = Path('../master_data/rve_run2.h5')
SPLIT         = 'train'   # 'train' | 'val' | 'test'
RVE_START     = 0         # first RVE index to process
N_RVE         = 5         # how many consecutive RVEs to generate
STEPS_PER_RVE = 100

OUTPUT_DIR  = Path('stress_evolution_gifs')
OUTPUT_DIR.mkdir(exist_ok=True)

FPS         = 15
DPI         = 120
STRESS_CMAP = 'jet'

In [3]:
# ── Animation builder ──────────────────────────────────────────────────────────
color_exx, color_eyy, color_exy = '#e41a1c', '#377eb8', '#4daf4a'

def make_animation(rve_idx: int) -> Path:
    """Build and save one stress-evolution animation for `rve_idx`."""

    # Load data
    start = rve_idx * STEPS_PER_RVE
    end   = start + STEPS_PER_RVE
    with h5py.File(MASTER_FILE, 'r') as f:
        strains   = f[f'{SPLIT}/x_global'][start:end]       # (T, 3)
        mat_phase = f[f'{SPLIT}/x_local'][start, :, :, 0]  # (96, 96)
        stresses  = f[f'{SPLIT}/y_local'][start:end]        # (T, 96, 96, 3)

    T     = len(strains)
    steps = np.arange(T)
    sxx_lim = (stresses[:,:,:,0].min(), stresses[:,:,:,0].max())
    syy_lim = (stresses[:,:,:,1].min(), stresses[:,:,:,1].max())
    sxy_lim = (stresses[:,:,:,2].min(), stresses[:,:,:,2].max())

    rve_label = rve_idx + 1

    # ── Figure layout ──────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 11), dpi=DPI)
    gs = fig.add_gridspec(
        2, 1,
        hspace=0.08,
        top=0.93, bottom=0.05, left=0.06, right=0.94,
    )
    gs_top = gs[0].subgridspec(1, 2, wspace=0.55)
    ax_mat    = fig.add_subplot(gs_top[0, 0])
    ax_strain = fig.add_subplot(gs_top[0, 1])

    gs_bot = gs[1].subgridspec(1, 3, wspace=0.45)
    ax_sxx = fig.add_subplot(gs_bot[0, 0])
    ax_syy = fig.add_subplot(gs_bot[0, 1])
    ax_sxy = fig.add_subplot(gs_bot[0, 2])

    # ── Material phase (static) ────────────────────────────────────────────────
    im_mat = ax_mat.imshow(
        mat_phase, origin='lower', cmap='viridis',
        vmin=0, vmax=1, aspect='equal',
    )
    fig.colorbar(im_mat, ax=ax_mat, fraction=0.046, pad=0.04)
    ax_mat.set_title('Material Phase\n(0 = soft, 1 = hard)', fontsize=12)
    ax_mat.set_xlabel('X Grid', fontsize=11)
    ax_mat.set_ylabel('Y Grid', fontsize=11)

    # ── Strain evolution (static lines + moving marker) ────────────────────────
    ax_strain.plot(steps, strains[:, 0], color=color_exx, lw=1.5, label=r'$\epsilon_{xx}$')
    ax_strain.plot(steps, strains[:, 1], color=color_eyy, lw=1.5, label=r'$\epsilon_{yy}$')
    ax_strain.plot(steps, strains[:, 2], color=color_exy, lw=1.5, label=r'$\epsilon_{xy}$')
    ax_strain.set_xlabel('Step', fontsize=12)
    ax_strain.set_ylabel(r'$\epsilon_M$', fontsize=14)
    ax_strain.set_title('Strain Evolution', fontsize=12)
    ax_strain.legend(fontsize=12, loc='upper left')
    ax_strain.set_xlim(-1, T)
    ax_strain.grid(True, lw=0.4, alpha=0.5)

    vline    = ax_strain.axvline(x=0, color='k', lw=1.0, ls='--', alpha=0.7)
    dot_exx, = ax_strain.plot(0, strains[0, 0], 'o', color=color_exx, ms=8, zorder=5)
    dot_eyy, = ax_strain.plot(0, strains[0, 1], 'o', color=color_eyy, ms=8, zorder=5)
    dot_exy, = ax_strain.plot(0, strains[0, 2], 'o', color=color_exy, ms=8, zorder=5)

    # ── Stress fields ──────────────────────────────────────────────────────────
    im_sxx = ax_sxx.imshow(
        stresses[0, :, :, 0], origin='lower', cmap=STRESS_CMAP,
        vmin=sxx_lim[0], vmax=sxx_lim[1], aspect='equal',
    )
    fig.colorbar(im_sxx, ax=ax_sxx, fraction=0.046, pad=0.04, label='[MPa]')
    ax_sxx.set_title(r'$\sigma_{xx}$', fontsize=14)
    ax_sxx.set_xlabel('X Grid', fontsize=11); ax_sxx.set_ylabel('Y Grid', fontsize=11)

    im_syy = ax_syy.imshow(
        stresses[0, :, :, 1], origin='lower', cmap=STRESS_CMAP,
        vmin=syy_lim[0], vmax=syy_lim[1], aspect='equal',
    )
    fig.colorbar(im_syy, ax=ax_syy, fraction=0.046, pad=0.04, label='[MPa]')
    ax_syy.set_title(r'$\sigma_{yy}$', fontsize=14)
    ax_syy.set_xlabel('X Grid', fontsize=11); ax_syy.set_ylabel('Y Grid', fontsize=11)

    im_sxy = ax_sxy.imshow(
        stresses[0, :, :, 2], origin='lower', cmap=STRESS_CMAP,
        vmin=sxy_lim[0], vmax=sxy_lim[1], aspect='equal',
    )
    fig.colorbar(im_sxy, ax=ax_sxy, fraction=0.046, pad=0.04, label='[MPa]')
    ax_sxy.set_title(r'$\sigma_{xy}$', fontsize=14)
    ax_sxy.set_xlabel('X Grid', fontsize=11); ax_sxy.set_ylabel('Y Grid', fontsize=11)

    def _title(frame):
        e = strains[frame]
        return (f'RVE {rve_label:03d} – Step {frame} | '
                f'Macro strain = [{e[0]:.5f}, {e[1]:.5f}, {e[2]:.5f}]')

    title_text = fig.suptitle(_title(0), fontsize=13, y=0.98)

    # ── Update function ────────────────────────────────────────────────────────
    def update(frame):
        im_sxx.set_data(stresses[frame, :, :, 0])
        im_syy.set_data(stresses[frame, :, :, 1])
        im_sxy.set_data(stresses[frame, :, :, 2])
        vline.set_xdata([frame, frame])
        dot_exx.set_data([frame], [strains[frame, 0]])
        dot_eyy.set_data([frame], [strains[frame, 1]])
        dot_exy.set_data([frame], [strains[frame, 2]])
        title_text.set_text(_title(frame))
        return im_sxx, im_syy, im_sxy, vline, dot_exx, dot_eyy, dot_exy, title_text

    anim = FuncAnimation(fig, update, frames=T, interval=1000 / FPS, blit=False)

    # ── Save ───────────────────────────────────────────────────────────────────
    if shutil.which('ffmpeg'):
        out_path = OUTPUT_DIR / f'rve_{rve_label:03d}.mp4'
        anim.save(str(out_path), writer=FFMpegWriter(fps=FPS), dpi=DPI)
    else:
        out_path = OUTPUT_DIR / f'rve_{rve_label:03d}.gif'
        anim.save(str(out_path), writer=PillowWriter(fps=FPS), dpi=DPI)

    plt.close(fig)
    return out_path

print('make_animation() ready.')

make_animation() ready.


In [4]:
# ── Generate N animations ──────────────────────────────────────────────────────
for i, rve_idx in enumerate(range(RVE_START, RVE_START + N_RVE)):
    out = make_animation(rve_idx)
    print(f'[{i+1}/{N_RVE}] saved: {out.name}')

[1/5] saved: rve_001.gif
[2/5] saved: rve_002.gif
[3/5] saved: rve_003.gif
[4/5] saved: rve_004.gif
[5/5] saved: rve_005.gif


In [5]:
# ── Inline preview: single frame (midpoint of first RVE) ──────────────────────
PREVIEW_RVE = RVE_START
start_p = PREVIEW_RVE * STEPS_PER_RVE

with h5py.File(MASTER_FILE, 'r') as f:
    strains_p   = f[f'{SPLIT}/x_global'][start_p:start_p + STEPS_PER_RVE]
    mat_phase_p = f[f'{SPLIT}/x_local'][start_p, :, :, 0]
    stresses_p  = f[f'{SPLIT}/y_local'][start_p:start_p + STEPS_PER_RVE]

T_p     = len(strains_p)
steps_p = np.arange(T_p)
STEP    = T_p // 2
sxx_lim_p = (stresses_p[:,:,:,0].min(), stresses_p[:,:,:,0].max())
syy_lim_p = (stresses_p[:,:,:,1].min(), stresses_p[:,:,:,1].max())
sxy_lim_p = (stresses_p[:,:,:,2].min(), stresses_p[:,:,:,2].max())
rve_label  = PREVIEW_RVE + 1

fig2 = plt.figure(figsize=(18, 11))
gs2 = fig2.add_gridspec(2, 1, hspace=0.08, top=0.93, bottom=0.05, left=0.06, right=0.94)
gs2_top = gs2[0].subgridspec(1, 2, wspace=0.55)
gs2_bot = gs2[1].subgridspec(1, 3, wspace=0.45)
ax2_mat    = fig2.add_subplot(gs2_top[0, 0])
ax2_strain = fig2.add_subplot(gs2_top[0, 1])
ax2_sxx    = fig2.add_subplot(gs2_bot[0, 0])
ax2_syy    = fig2.add_subplot(gs2_bot[0, 1])
ax2_sxy    = fig2.add_subplot(gs2_bot[0, 2])

# Material phase
im2 = ax2_mat.imshow(mat_phase_p, origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
fig2.colorbar(im2, ax=ax2_mat, fraction=0.046, pad=0.04)
ax2_mat.set_title('Material Phase\n(0 = soft, 1 = hard)', fontsize=12)
ax2_mat.set_xlabel('X Grid', fontsize=11); ax2_mat.set_ylabel('Y Grid', fontsize=11)

# Strain evolution
ax2_strain.plot(steps_p, strains_p[:, 0], color=color_exx, lw=1.5, label=r'$\epsilon_{xx}$')
ax2_strain.plot(steps_p, strains_p[:, 1], color=color_eyy, lw=1.5, label=r'$\epsilon_{yy}$')
ax2_strain.plot(steps_p, strains_p[:, 2], color=color_exy, lw=1.5, label=r'$\epsilon_{xy}$')
ax2_strain.axvline(x=STEP, color='k', lw=1.0, ls='--', alpha=0.7)
ax2_strain.plot(STEP, strains_p[STEP, 0], 'o', color=color_exx, ms=8)
ax2_strain.plot(STEP, strains_p[STEP, 1], 'o', color=color_eyy, ms=8)
ax2_strain.plot(STEP, strains_p[STEP, 2], 'o', color=color_exy, ms=8)
ax2_strain.set_title('Strain Evolution', fontsize=12)
ax2_strain.set_xlabel('Step', fontsize=12)
ax2_strain.set_ylabel(r'$\epsilon_M$', fontsize=14)
ax2_strain.legend(fontsize=12, loc='upper left')
ax2_strain.grid(True, lw=0.4, alpha=0.5)
ax2_strain.set_xlim(-1, T_p)

# Stress fields
for ax, data, title, lim in [
    (ax2_sxx, stresses_p[STEP,:,:,0], r'$\sigma_{xx}$', sxx_lim_p),
    (ax2_syy, stresses_p[STEP,:,:,1], r'$\sigma_{yy}$', syy_lim_p),
    (ax2_sxy, stresses_p[STEP,:,:,2], r'$\sigma_{xy}$', sxy_lim_p),
]:
    im_ = ax.imshow(data, origin='lower', cmap=STRESS_CMAP, vmin=lim[0], vmax=lim[1], aspect='equal')
    fig2.colorbar(im_, ax=ax, fraction=0.046, pad=0.04, label='[MPa]')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('X Grid', fontsize=11); ax.set_ylabel('Y Grid', fontsize=11)

e = strains_p[STEP]
fig2.suptitle(
    f'RVE {rve_label:03d} – Step {STEP} | '
    f'Macro strain = [{e[0]:.5f}, {e[1]:.5f}, {e[2]:.5f}]',
    fontsize=13, y=0.98,
)
plt.show()

C:\Users\badesia\AppData\Local\Temp\ipykernel_16584\3074780986.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
